In [ ]:
## Packages Import
import copy
import numpy             as np
import matplotlib.pyplot as plt

In [ ]:
class VLMPanel:
    """
    Vortex Lattice Method panel with horseshoe vortex
    """
    #--------------------#
    #   Initialization   #
    #--------------------#
    def __init__(self, p, b):
        """
        Panel is of quadrilateral shape and vertices must be ordered from root
        position on leading-edge proceeding in counterclockwise direction. 
        """
        # Wing geometry
        self.span = b       # wing span [m]

        # Panel geometry
        self.pnt    = p        # panel vertices
        self.ctr    = None     # control point
        self.normal = None     # normal direction
        self.chord  = None     # chord length [m]
        self.width  = None     # pannel width [m]
        self.area   = None     # panel area   [m**2]

        # Vortex parameters
        self.vrt = None     # horseshoe vortex points
        
        self._get_panel_geom()
        self._get_vortex_geom(20*b)


    #-------------#
    #   Methods   #
    #-------------#
    def _get_panel_geom(self):
        """
        Compute panel geometric parameters such as control point position, normal
        versor, chord length, width and area, starting from panel's vertices.
        """
        p = self.pnt

        # Compute representative panel's geometric parameters
        c_avg = 0.5 * (p[2][0] + p[3][0] - p[0][0] - p[1][0])  # average chord length
        w_avg = 0.5 * (p[1][2] + p[2][2] - p[0][2] - p[3][2])  # average width

        # Compute position of control point
        cp = np.mean(p, axis=0)
        cp[0] += (0.25 * c_avg)     # three-quarter line

        # Compute normal versor and panel surface
        ai = np.zeros(3)
        for i, pi in enumerate(p):
            qi = p[(i + 1) % len(p)]
            ai += np.cross(pi, qi)

        av = 0.5 * ai
        a = np.linalg.norm(av)
        if a > 0.0:
            n = av / a
        else:
            # Degenerate cells with zero area
            print(f"Degenerate panel {i} with {len(p)} vertices")

        self.ctr    = cp
        self.normal = n
        self.chord  = c_avg
        self.width  = w_avg
        self.area   = a

    def _get_vortex_geom(self, d):
        """
        Compute horseshoe vortex's vertices position starting from panel's vertices.
        Horseshoe vortex is approximated by a ring vortex with negligible contribution
        by the wake vortex.

        Input:
        d -> wake vortex distance from wing
        """
        p = self.pnt

        v = []
        for i, pi in enumerate(p):
            vi = np.copy(pi)
            if i in [0, 1]:
                # Bound vortex lies on the quater chord line
                vi[0] += (0.25 * self.chord)
            elif i in [2, 3]:
                # Wake vortex lies far behind the wing
                vi[0] += d   # HP: trailing vortices parallel to x-axis
            v.append(vi)

        self.vrt = v

In [ ]:
class VLMSolver:
    """
    Vortex Lattice Method solver
    """
    #--------------------#
    #   Initialization   #
    #--------------------#
    def __init__(self, b, c, theta, gamma, lamb, sym,
                u_inf, n):
        # Wing geometry
        self.span     = b       # wing span                 [m]
        self.chord    = c       # chord lengths             [m]
        self.incid    = theta   # angle of incidence        [rad]
        self.dihedron = gamma   # dihedral angle            [rad]
        self.sweep    = lamb    # quarter-chord sweep angle [rad]
        self.symmetric = sym    # symmetric wing

        # Flow properties
        self.U = u_inf      # inflow velocity [m/s]

        # Discretization
        self.N = n              # number of panels
        self.panels = None      # panel objects

        # Linear system
        self.A = None       # coefficient matrix
        self.b = None       # right hand side

    
    #-------------#
    #   Methods   #
    #-------------#
    def _unit_rot(self, v, theta, ax, inverse=False):
        """
        Elemental clockwise rotation of the reference system axes.
        
        Input:
        v     -> vector componets - shape: (N, 3)
        theta -> rotation angle - clockwise rotation: theta>0
        ax    -> rotation axis index
        """
        # Assemble rotation matrix
        rot = np.zeros((3, 3))  # rotation matrix
        # diagonal elements
        rot[ax, ax] = 1.0
        rot[ax-1, ax-1] = np.cos(theta)
        rot[ax-2, ax-2] = np.cos(theta)
        # extra-diagonal elements
        rot[ax-2, ax-1] = -np.sin(theta)
        rot[ax-1, ax-2] = np.sin(theta)

        if inverse:
            # Apply inverse rotation
            rot = rot.T

        # Apply rotation
        v_rot = np.sum(rot[None, :, :] * v[:, None, :], axis=2)
        return v_rot


    def _build_mesh(self):
        """
        Discretization of the wing by a finite number of lattices.
        """
        n     = self.N
        b     = self.span
        theta = self.incid
        lamb  = self.sweep
        gamma = self.dihedron
        c     = self.chord

        # Leading-edge length of the semi-wing
        le = (0.5 * b) / (np.cos(lamb) * np.cos(gamma))
        if self.symmetric:
            s_min, s_max = 0.0, le
        else:
            s_min, s_max = -le, le

        # Spanwise wing discretization
        s = np.linspace(s_min, s_max, (n+1))    # leading-edge coordinates

        # Change reference system for leading-edge coordinates
        le_wing = np.column_stack((np.zeros_like(s), np.zeros_like(s), s))      # wing frame
        le_i  = self._unit_rot(le_wing, (-theta * np.cos(lamb)), 2, True)   # incidence rotation (about spanwise axis)
        le_ii = self._unit_rot(le_i, (gamma * np.cos(lamb)), 0, True)       # dihedral rotation (about chordwise axis)
        le_body = self._unit_rot(le_ii, -lamb, 1, True)                         # body frame
        
        # Translate leading-edge coordinates into trailing-edge points
        c_wing = np.column_stack((c, np.zeros_like(c), np.zeros_like(c)))       # wing frame
        c_i  = self._unit_rot(c_wing, (-theta * np.cos(lamb)), 2, True)     # incidence rotation (about spanwise axis)
        c_ii = self._unit_rot(c_i, (gamma * np.cos(lamb)), 0, True)         # dihedral rotation (about chordwise axis)
        c_body = self._unit_rot(c_ii, -lamb, 1, True)                           # body frame
        te_body = le_body + np.column_stack((c_body[:, :2], np.zeros_like(c)))  # rigid translation of leading-edge coordinates
    
        panels = []     # list of panel objects
        for i in range(n):
            pnt = [le_body[i],
                le_body[i+1],
                te_body[i+1],
                te_body[i]]    # ordered panel vertices

            panels.append(VLMPanel(p=pnt, b=b))

        self.panels = panels

In [ ]:
## Utilities

def chord_fn(n, c0, shape="rectangular"):
    """
    Return chord length distribution along spanwise direction according to the
    desired planform shape.

    Input:
        n     -> number of grid points
        c0    -> root chord length
        shape -> wing planform
    """
    match shape:
        case "rectangular":
            c = c0 * np.ones(n)
        # case "elliptical":
        #     c = c0 * np.sqrt(1 - (z/s)**2)
    return c

import numpy as np

def normalize(v):
    """
    Input:
        v -> vector
    """
    # Cast input object into array of floats
    v = np.asarray(v, dtype=float)
    # Compute vector norm 
    norm = np.linalg.norm(v)
    if norm == 0.0:
        raise ValueError("Cannot normalize a zero vector")
    return v / norm

Geometric parameters definitons:
- Incidence $\theta$ is the angle between the wing root chord line direction and the vertical XZ-plane, positive if the trailing edge has positive Y coordinate
- Sweep $\Lambda$ is the angle between the wing spanwise direction and the side YZ-plane, positive if the wing tip has positive X coordinate
- Dihedral $\Gamma$ is the angle between the wing spanwise direction and the horizontal XZ-plane, positive if the wing tip has positive Y coordinate

In [ ]:
## Parameters Definition

# Wing geometry
B = 1.0         # wing span                 [m]
AR = 5.0        # aspect ration             [-]
THETA  = 20.0   # incidence angle           [deg]
GAMMA  = 10.0   # dihedral angle            [deg]
LAMBDA = 40.0   # quarter-chord sweep angle [deg]

SYM = True   # symmetric wing configuration

# Flow properties
U = 1.0     # inflow velocity [m/s]

# Wing discretization
N = 2       # number of panels in spanwise direction

In [ ]:
## TEST

# Untapered wing (constant chord length distribution)
c0 = B / AR     # chord length [m]

# Wing discretization and attitude
vlm = VLMSolver(b=B,
                c=chord_fn((N+1), c0),
                theta=np.deg2rad(THETA),
                gamma=np.deg2rad(GAMMA),
                lamb=np.deg2rad(LAMBDA),
                sym=SYM,
                u_inf=np.array([U, 0.0, 0.0]),
                n=N)
vlm._build_mesh()


fig_test = plt.figure(figsize=(18, 5))
ax1 = fig_test.add_subplot(1, 3, 1)
ax2 = fig_test.add_subplot(1, 3, 2)
ax3 = fig_test.add_subplot(1, 3, 3)


for i, panel in enumerate(vlm.panels):
    pnt    = copy.copy(panel.pnt)
    ctr    = copy.copy(panel.ctr)
    normal = copy.copy(panel.normal)
    vrt    = copy.copy(panel.vrt)
    # Close the polygon shape
    pnt.append(pnt[0])
    pnt_plt = np.array(pnt)
    vrt.append(vrt[0])
    vrt_plt = np.array(vrt)

    if i == 0:
        # Store root panel vertices
        pnt_root = pnt

        # Plot reference direction
        ax1.axline(pnt_plt[0, :2:],  slope=np.tan(np.deg2rad(90-LAMBDA)), c='k', lw=2, ls='--', label="ref")
        ax2.axline(pnt_plt[0, 1::],  slope=np.tan(np.deg2rad(90-GAMMA)),  c='k', lw=2, ls='--', label="ref")
        ax3.axline(pnt_plt[0, ::-2], slope=np.tan(np.deg2rad(THETA)),     c='k', lw=2, ls='--', label="ref")
    elif i == (N - 1):
        # Store tip panel vertices
        pnt_tip = pnt

    # Select plotting color
    clr = ax1._get_lines.get_next_color()

    # Plot lattice
    # lateral view
    ax1.plot(pnt_plt[:, 0], pnt_plt[:, 2], '-o',     color=clr, label=f"panel #{i+1}")
    ax1.quiver(ctr[0], ctr[2], normal[0], normal[2], color=clr)
    # frontal view
    ax2.plot(pnt_plt[:, 1], pnt_plt[:, 2], '-o',     color=clr, label=f"panel #{i+1}")
    ax2.quiver(ctr[1], ctr[2], normal[1], normal[2], color=clr)
    # top view
    ax3.plot(pnt_plt[:, 0], pnt_plt[:, 1], '-o',     color=clr, label=f"panel #{i+1}")
    ax3.quiver(ctr[0], ctr[1], normal[0], normal[1], color=clr)

ax1.set_xlabel("X [m]")
ax1.set_ylabel("Z [m]")
ax1.axis('equal')
ax1.legend()
ax1.grid(True)
ax2.set_xlabel("Y [m]")
ax2.set_ylabel("Z [m]")
ax2.axis('equal')
ax2.legend()
ax2.grid(True)
ax3.set_xlabel("X [m]")
ax3.set_ylabel("Y [m]")
ax3.axis('equal')
ax3.legend()
ax3.grid(True)

# Compute wing attitude
dir_chord = normalize(pnt_root[3] - pnt_root[0])    # chordwise direction
dir_span = normalize(pnt_tip[1] - pnt_root[0])      # spanwise direction

print("Computed wing geometry:")
print(f"    - Incidence angle = {np.rad2deg(np.atan2(dir_chord[1], dir_chord[0])):.2f}°")
print(f"    - Dihedral angle = {np.rad2deg(np.atan2(dir_span[1], dir_span[2])):.2f}°")
print(f"    - Sweep angle = {np.rad2deg(np.atan2(dir_span[0], dir_span[2])):.1f}°")

del pnt, pnt_plt, ctr, normal, vrt, vrt_plt